# Chapter 75: Network Anomaly Detection

**Volume 4 — Security Operations**

**The attacker does not trigger any signature.** No known bad IP. No malicious port.
Their traffic is technically valid. It just does not look like *your* network's normal.

Anomaly detection inverts the problem: instead of asking
*"does this match something bad?"* it asks *"does this match our normal?"*

### What You Will Build — 3 Detection Layers

| Demo | Method | Catches |
|------|--------|---------|
| **1. Z-Score Baseline** | Stratified statistics per hour+weekday | DDoS, sudden traffic spikes |
| **2. Entropy Analyzer** | Shannon entropy on traffic distribution | Port scans, IP scans, DNS tunneling |
| **3. Autoencoder** | 11-feature pure-numpy neural net | Multi-dimensional anomalies |
| **4. EWMA Trend Detector** | Exponential moving average trends | Slow-burn exfiltration over hours |
| **5. Full Fusion Pipeline** | All 3 layers + LLM synthesis | Any of the above, higher confidence |

### The Core Insight
> **"Normal" is not a single number. It is a multi-dimensional profile:
> how much traffic (volumetric), who the device talks to (behavioral),
> and which protocols it uses (entropic). An anomaly is a deviation
> in one or more of these dimensions — and the attack you miss is often
> the one that deviates in only one, subtle dimension.**

In [ ]:
# pip install anthropic
# Network Anomaly Detection — pure Python + numpy, no ML frameworks needed!
# numpy is pre-installed in Colab; for local: pip install numpy

import os, math, random, time
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

try:
    import numpy as np
    HAS_NUMPY = True
    print("numpy available - autoencoder fully functional")
except ImportError:
    HAS_NUMPY = False
    print("numpy not found - autoencoder runs in simulation mode")
    print("Install: pip install numpy")

# ── Anthropic client ───────────────────────────────────────────────────────────
api_key = os.environ.get("ANTHROPIC_API_KEY", "")
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("LLM analyst: ACTIVE (Anthropic API)")
else:
    print("LLM analyst: SIMULATION MODE (set ANTHROPIC_API_KEY for real analysis)")

def llm_explain(prompt: str, max_tokens: int = 150) -> str:
    '''Call Anthropic API or return a simulation response.'''
    if USE_API:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251011",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text.strip()
    # Simulation: context-aware responses
    p = prompt.lower()
    if "ddos" in p or "spike" in p or "packets" in p:
        return ("MITRE T1498 (Network DoS). High outbound packet rate with "
                "diverse sources = volumetric attack. Recommend: activate "
                "upstream scrubbing, apply rate limits, notify ISP.")
    if "port scan" in p or "destination port" in p:
        return ("MITRE T1046 (Network Service Discovery). High port entropy "
                "from single source = port scan. Recommend: block source IP, "
                "check if any port accepted a connection, harden exposed services.")
    if "exfil" in p or "outbound" in p or "asymmet" in p:
        return ("MITRE T1048 (Exfiltration). High outbound bytes to rare "
                "destination with low inbound = data exfiltration. Recommend: "
                "block destination, isolate source host, preserve memory image.")
    if "dns" in p or "entropy" in p or "tunnel" in p:
        return ("MITRE T1071.004 (DNS tunneling). High query-name entropy and "
                "unusual subdomain length = data encoded in DNS. Recommend: "
                "DNS sinkhole, audit querying hosts for malware.")
    if "slow" in p or "gradual" in p or "trend" in p:
        return ("MITRE T1048 (slow exfiltration). Gradual volume increase over "
                "multiple hours stays below per-window thresholds. Recommend: "
                "isolate device, forensic investigation, check scheduled tasks.")
    return ("Multi-dimensional anomaly detected. Traffic pattern deviates from "
            "learned normal in multiple features. Recommend analyst investigation.")

# Shared constants for all demos
FEATURES = ["bytes_out","bytes_in","pkts_out","pkts_in","flows_out",
            "uniq_dst_ip","uniq_dst_port","pct_tcp","pct_udp",
            "entropy_dst_ip","entropy_dst_port"]

print(f"\nAll {len(FEATURES)} traffic features ready.")
print("Starting with Demo 1: Statistical Baseline + Z-Score Detector")


## Demo 1: Statistical Baseline + Z-Score Detector

Network traffic follows **strong seasonal patterns**: 50,000 flows/minute on a Monday
afternoon is normal. At 3am Sunday, 200 flows/minute is normal.
A single mean over 30 days is wrong for both.

**The right approach: stratified baselines — 168 windows (24 hours x 7 days)**

Each (hour, weekday) bucket gets its own mean and standard deviation,
trained only on data from that exact time slot.

**Z-score formula:**
```
z = (observed - mean) / std_dev

z > 2.5  ->  low-confidence anomaly   (log and monitor)
z > 3.5  ->  medium-confidence alert  (analyst queue)
z > 5.0  ->  high-confidence alert    (immediate action)
```

*Network analogy: Z-score is like BGP MED comparison.
You don't compare a Monday-morning route to a Sunday-night route —
you compare Monday morning to your baseline of Monday mornings.
Same idea: same time slot, same baseline.*

In [ ]:
# ── Demo 1: Statistical Baseline + Z-Score Detector ──────────────────────────

@dataclass
class TrafficWindow:
    '''One 5-minute traffic summary for a device.'''
    device: str
    hour: int        # 0-23
    weekday: int     # 0=Mon ... 6=Sun
    bytes_out: float
    packets_out: float
    flows_out: float
    unique_dst_ips: float

class StratifiedBaseline:
    '''
    Builds per-(hour, weekday) statistical baselines for each metric.
    Z-score anomaly detection against the correct time-slot baseline.
    '''

    def __init__(self):
        # Dict: (hour, weekday) -> {"metric": [values...]}
        self.buckets: Dict[Tuple, Dict[str, list]] = defaultdict(lambda: defaultdict(list))
        # Computed stats: (hour, weekday) -> {"metric": (mean, std)}
        self.stats: Dict[Tuple, Dict[str, Tuple[float, float]]] = {}

    def train(self, windows: List[TrafficWindow]):
        '''Ingest historical windows into per-time-slot buckets.'''
        for w in windows:
            key = (w.hour, w.weekday)
            self.buckets[key]["bytes_out"].append(w.bytes_out)
            self.buckets[key]["packets_out"].append(w.packets_out)
            self.buckets[key]["flows_out"].append(w.flows_out)
            self.buckets[key]["unique_dst_ips"].append(w.unique_dst_ips)

    def fit(self):
        '''Compute mean and std for each (hour, weekday) bucket.'''
        for key, metrics in self.buckets.items():
            self.stats[key] = {}
            for metric, values in metrics.items():
                if len(values) < 2:
                    self.stats[key][metric] = (values[0] if values else 0.0, 1.0)
                    continue
                mean = sum(values) / len(values)
                variance = sum((v - mean) ** 2 for v in values) / len(values)
                std = max(math.sqrt(variance), 1e-6)  # avoid div-by-zero
                self.stats[key][metric] = (mean, std)
        print(f"[Baseline] Trained on {len(self.stats)} time-slot buckets")

    def z_score(self, metric: str, value: float,
                hour: int, weekday: int) -> float:
        '''Compute z-score for a metric value against the correct time-slot.'''
        key = (hour, weekday)
        if key not in self.stats or metric not in self.stats[key]:
            return 0.0
        mean, std = self.stats[key][metric]
        return (value - mean) / std

    def score_window(self, w: TrafficWindow) -> dict:
        '''
        Score all metrics in a traffic window.
        Returns dict with z-score per metric + max z-score.
        '''
        metrics = {
            "bytes_out": w.bytes_out,
            "packets_out": w.packets_out,
            "flows_out": w.flows_out,
            "unique_dst_ips": w.unique_dst_ips,
        }
        z_scores = {
            m: self.z_score(m, v, w.hour, w.weekday)
            for m, v in metrics.items()
        }
        max_z = max(abs(z) for z in z_scores.values())
        return {"z_scores": z_scores, "max_z": round(max_z, 2)}

def z_severity(max_z: float) -> str:
    if max_z >= 5.0: return "HIGH"
    if max_z >= 3.5: return "MEDIUM"
    if max_z >= 2.5: return "LOW"
    return "NORMAL"

# ── Simulate 90 days of normal baseline data ──────────────────────────────────
random.seed(42)
training_windows = []
for day in range(90):
    weekday = day % 7
    for hour in range(24):
        # Business hours: higher traffic Mon-Fri 8-18
        is_business = (weekday < 5 and 8 <= hour < 18)
        base_bytes  = 50_000_000 if is_business else 5_000_000
        base_pkts   = 50_000 if is_business else 5_000
        base_flows  = 10_000 if is_business else 1_000
        base_dsts   = 500 if is_business else 50

        training_windows.append(TrafficWindow(
            device="db-server-01", hour=hour, weekday=weekday,
            bytes_out  = base_bytes  * random.uniform(0.8, 1.2),
            packets_out= base_pkts   * random.uniform(0.8, 1.2),
            flows_out  = base_flows  * random.uniform(0.8, 1.2),
            unique_dst_ips = base_dsts * random.uniform(0.8, 1.2),
        ))

baseline = StratifiedBaseline()
baseline.train(training_windows)
baseline.fit()

# ── Score live traffic windows ────────────────────────────────────────────────
print("=" * 60)
print("Z-SCORE ANOMALY DETECTOR — LIVE TRAFFIC ANALYSIS")
print("=" * 60)

test_windows = [
    TrafficWindow("db-server-01", 14, 1,  52_000_000,  51_000, 10_200, 510),  # normal Mon 14:00
    TrafficWindow("db-server-01",  3, 6,   4_800_000,   4_900,    950,  48),  # normal Sun 03:00
    TrafficWindow("db-server-01",  3, 6, 980_000_000, 950_000,  9_500,  48),  # DDoS! Sun 3am spike
    TrafficWindow("db-server-01", 14, 1, 820_000_000, 780_000, 82_000, 420),  # Mon afternoon DDoS
]

labels = [
    "Mon 14:00 - normal business hours",
    "Sun 03:00 - normal overnight",
    "Sun 03:00 - DDoS (200x normal!)",
    "Mon 14:00 - DDoS on busy link",
]

for w, label in zip(test_windows, labels):
    result = baseline.score_window(w)
    sev = z_severity(result["max_z"])
    print(f"\n[{sev:6}] {label}")
    print(f"  max_z={result['max_z']:.1f} | "
          + " | ".join(f"{k}={v:.1f}" for k, v in result["z_scores"].items()))
    if sev in ("HIGH", "MEDIUM"):
        top_metric = max(result["z_scores"], key=lambda k: abs(result["z_scores"][k]))
        analysis = llm_explain(
            f"Z-score anomaly on db-server-01: metric '{top_metric}' "
            f"scored z={result['z_scores'][top_metric]:.1f} at {w.hour:02d}:00. "
            f"This is {w.bytes_out/1e6:.0f} MB outbound vs "
            f"normal ~{baseline.stats.get((w.hour,w.weekday),{}).get('bytes_out',(0,0))[0]/1e6:.0f} MB. "
            f"Possible DDoS. MITRE technique? Action? Under 60 words.",
            max_tokens=80
        )
        print(f"  LLM: {analysis}")

print("\n[Z-Score] Fast, interpretable, per-metric thresholds.")
print("Best for: volumetric anomalies (DDoS, traffic spikes, sudden drops).")


## Demo 2: Entropy Analyzer — Detecting Distribution Anomalies

Z-scores detect volume changes. But some attacks **don't change total volume** —
they change the *distribution* of traffic.

A port scan from one IP to 65,000 ports adds little total volume,
but the destination-port distribution goes from concentrated to maximally diverse.

**Shannon Entropy** measures this diversity:
```
H = -sum( p(x) * log2(p(x)) )

H = 0.0   ->  all traffic to ONE destination (perfectly concentrated)
H = 1.0   ->  traffic spread across ALL possible destinations (maximum diversity)
```

| Pattern | Port Entropy | IP Entropy | What It Means |
|---------|-------------|-----------|---------------|
| Normal web server | LOW (80, 443 only) | LOW (few clients) | Healthy |
| Port scan | HIGH (all ports) | LOW (one source) | T1046 recon |
| IP scan / host discovery | LOW (same port) | HIGH (all IPs) | T1046 recon |
| DNS tunneling | n/a | HIGH query entropy | T1071.004 |
| DDoS botnet | LOW (one target port) | HIGH (many sources) | T1498 |

*Network analogy: entropy is like route table diversity.
Normally your routers have routes to a few prefixes.
If a host suddenly generates traffic to 50,000 unique /32 destinations,
the entropy of that routing table change tells you something is wrong.*

In [ ]:
# ── Demo 2: Entropy Analyzer ──────────────────────────────────────────────────

def shannon_entropy(counts: dict) -> float:
    '''
    Compute normalized Shannon entropy of a frequency distribution.
    Input: dict of {value: count}
    Output: 0.0 (all same) to 1.0 (maximally diverse)
    '''
    total = sum(counts.values())
    if total == 0 or len(counts) == 0:
        return 0.0
    entropy = 0.0
    for count in counts.values():
        if count > 0:
            p = count / total
            entropy -= p * math.log2(p)
    # Normalize to 0-1 by dividing by max possible entropy
    max_entropy = math.log2(len(counts)) if len(counts) > 1 else 1.0
    return entropy / max_entropy if max_entropy > 0 else 0.0

@dataclass
class FlowRecord:
    '''Simulated NetFlow record.'''
    src_ip: str
    dst_ip: str
    dst_port: int
    protocol: str
    bytes_out: int
    query_name: str = ""  # for DNS records

class EntropyAnalyzer:
    '''
    Analyzes entropy of traffic distributions in 5-minute windows.
    Detects: port scans, IP scans, DNS tunneling.
    '''

    def __init__(self, port_threshold: float = 0.60,
                       dst_ip_threshold: float = 0.70,
                       dns_entropy_threshold: float = 0.75):
        self.port_thr  = port_threshold
        self.dst_thr   = dst_ip_threshold
        self.dns_thr   = dns_entropy_threshold

    def analyze_flows(self, src_ip: str,
                      flows: List[FlowRecord]) -> dict:
        '''
        Compute entropy metrics for all flows from a source IP.
        Returns: {entropy scores, alerts}
        '''
        # Count destinations and ports
        dst_port_counts: Dict[int, int] = defaultdict(int)
        dst_ip_counts:   Dict[str, int] = defaultdict(int)
        dns_name_lengths: List[int] = []

        for f in flows:
            if f.src_ip != src_ip:
                continue
            dst_port_counts[f.dst_port] += 1
            dst_ip_counts[f.dst_ip] += 1
            if f.protocol == "dns" and f.query_name:
                # Measure subdomain length as proxy for DNS tunneling content
                subdomain = f.query_name.split(".")[0] if "." in f.query_name else f.query_name
                dns_name_lengths.append(len(subdomain))

        port_entropy  = shannon_entropy(dst_port_counts)
        dst_ip_entropy = shannon_entropy(dst_ip_counts)

        # DNS: high average subdomain length suggests encoded data
        avg_dns_len = (sum(dns_name_lengths) / len(dns_name_lengths)
                       if dns_name_lengths else 0)
        dns_alert = avg_dns_len > 20  # subdomains >20 chars = suspicious

        alerts = []
        if port_entropy >= self.port_thr:
            alerts.append({
                "type": "PORT_SCAN",
                "mitre": "T1046",
                "entropy": round(port_entropy, 3),
                "unique_ports": len(dst_port_counts),
            })
        if dst_ip_entropy >= self.dst_thr:
            alerts.append({
                "type": "IP_SCAN",
                "mitre": "T1046",
                "entropy": round(dst_ip_entropy, 3),
                "unique_ips": len(dst_ip_counts),
            })
        if dns_alert:
            alerts.append({
                "type": "DNS_TUNNEL",
                "mitre": "T1071.004",
                "avg_subdomain_len": round(avg_dns_len, 1),
                "query_count": len(dns_name_lengths),
            })

        return {
            "src_ip": src_ip,
            "port_entropy": round(port_entropy, 3),
            "dst_ip_entropy": round(dst_ip_entropy, 3),
            "avg_dns_len": round(avg_dns_len, 1),
            "alerts": alerts,
        }

# ── Simulate different traffic scenarios ──────────────────────────────────────
print("=" * 60)
print("ENTROPY ANALYZER — TRAFFIC DISTRIBUTION ANALYSIS")
print("=" * 60)

analyzer = EntropyAnalyzer()

# Scenario 1: Normal web server traffic (concentrated ports and destinations)
normal_flows = [
    FlowRecord("10.0.1.5", "8.8.8.8", 443, "tcp", 15000),
    FlowRecord("10.0.1.5", "1.1.1.1", 443, "tcp", 8000),
    FlowRecord("10.0.1.5", "8.8.8.8", 80,  "tcp", 2000),
    FlowRecord("10.0.1.5", "8.8.4.4", 443, "tcp", 12000),
    FlowRecord("10.0.1.5", "8.8.8.8", 443, "tcp", 9000),
]

# Scenario 2: Port scan (one IP, many ports)
port_scan_flows = [
    FlowRecord("185.220.1.100", "10.0.0.5", port, "tcp", 60)
    for port in [22, 23, 25, 53, 80, 110, 135, 139, 143, 443,
                 445, 1433, 1521, 3306, 3389, 5432, 5900, 8080, 8443, 9200]
]

# Scenario 3: IP scan / host discovery (same port, many destinations)
ip_scan_flows = [
    FlowRecord("185.220.1.100", f"10.0.{subnet}.{host}", 22, "tcp", 60)
    for subnet in range(5)
    for host in range(20)
]

# Scenario 4: DNS tunneling (long random subdomains)
import hashlib
dns_tunnel_flows = [
    FlowRecord("10.0.2.33", "8.8.8.8", 53, "dns", 120,
               query_name=f"{hashlib.md5(str(i).encode()).hexdigest()[:24]}.attacker.com")
    for i in range(50)
]

# Scenario 5: Normal DNS (short meaningful names)
normal_dns = [
    FlowRecord("10.0.1.5", "8.8.8.8", 53, "dns", 80, query_name=q)
    for q in ["google.com", "microsoft.com", "github.com", "slack.com",
              "zoom.us", "aws.amazon.com", "api.github.com"]
]

scenarios = [
    ("10.0.1.5",      "Normal web traffic",      normal_flows + normal_dns),
    ("185.220.1.100", "Port scan (20 ports)",     port_scan_flows),
    ("185.220.1.100", "IP scan (100 hosts)",      ip_scan_flows),
    ("10.0.2.33",     "DNS tunneling",            dns_tunnel_flows),
]

for src_ip, label, flows in scenarios:
    result = analyzer.analyze_flows(src_ip, flows)
    status = "ALERT" if result["alerts"] else "NORMAL"
    print(f"\n[{status:6}] {label}")
    print(f"  port_entropy={result['port_entropy']:.3f}  "
          f"dst_ip_entropy={result['dst_ip_entropy']:.3f}  "
          f"avg_dns_len={result['avg_dns_len']:.1f}")

    for alert in result["alerts"]:
        print(f"  [!] {alert['type']} ({alert['mitre']})  "
              + " | ".join(f"{k}={v}" for k,v in alert.items()
                           if k not in ("type","mitre")))
        atype = alert["type"].lower().replace("_", " ")
        analysis = llm_explain(
            f"Entropy alert: {atype} from {src_ip}. "
            f"Port entropy={result['port_entropy']:.2f}, "
            f"IP entropy={result['dst_ip_entropy']:.2f}. "
            f"MITRE {alert['mitre']}. Recommended action? Under 50 words.",
            max_tokens=70
        )
        print(f"  LLM: {analysis}")

print("\n[EntropyAnalyzer] Best for: scans, tunneling, DGA — no volume change needed.")


## Demo 3: Autoencoder — Multi-Dimensional Anomaly Detection

Z-scores work per metric. Entropy works per distribution.
But some attacks only look anomalous **in combination**:

- Outbound bytes: slightly above normal (not enough for z-score alert)
- Destination IP: slightly unusual (not enough alone)
- Time of day: 2am (slightly odd but not alarming)
- All three together: data exfiltration — clearly anomalous

An **Autoencoder** learns the multi-dimensional normal pattern and detects
deviations across *any combination* of the 11 traffic features simultaneously.

**How it works:**
```
Normal traffic (11 features) -> Encoder -> 5 latent dims -> Decoder -> Reconstructed (11 features)
                                                                           |
                                                              Reconstruction error = LOW (model knows this)

Anomaly traffic (11 features) -> Encoder -> 5 latent dims -> Decoder -> Reconstructed (11 features)
                                                                           |
                                                              Reconstruction error = HIGH (model confused)
```

The **threshold** is set at the 99th percentile of reconstruction errors
on clean training data. Anything above = anomaly.

*Analogy: the autoencoder is like a network engineer who memorized every
normal config. When she tries to describe an attacker's modified config
from memory — the mismatch between her description and reality IS the alert.*

In [ ]:
# ── Demo 3: Pure-Python Autoencoder ──────────────────────────────────────────

if HAS_NUMPY:
    # ── Minimal autoencoder: 11 features -> 5 latent -> 11 reconstructed ─────
    class SimpleAutoencoder:
        '''
        3-layer autoencoder trained on normal traffic to detect anomalies.
        Architecture: input(11) -> encoder(5) -> decoder(11)
        Loss: Mean Squared Error (MSE) reconstruction error
        '''

        def __init__(self, input_dim=11, latent_dim=5, lr=0.01):
            rng = np.random.default_rng(42)
            self.lr = lr
            # Encoder: 11 -> 5
            self.W_enc = rng.normal(0, 0.1, (input_dim, latent_dim))
            self.b_enc = np.zeros(latent_dim)
            # Decoder: 5 -> 11
            self.W_dec = rng.normal(0, 0.1, (latent_dim, input_dim))
            self.b_dec = np.zeros(input_dim)
            self.threshold = None

        def _relu(self, x):
            return np.maximum(0, x)

        def _sigmoid(self, x):
            return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

        def encode(self, x):
            return self._relu(x @ self.W_enc + self.b_enc)

        def decode(self, z):
            return self._sigmoid(z @ self.W_dec + self.b_dec)

        def reconstruct(self, x):
            return self.decode(self.encode(x))

        def mse(self, x, x_hat):
            return float(np.mean((x - x_hat) ** 2))

        def train_step(self, X):
            '''One gradient descent step on batch X.'''
            Z = self.encode(X)
            X_hat = self.decode(Z)
            loss = self.mse(X, X_hat)
            # Simplified backprop (analytical gradient of MSE + sigmoid)
            grad_out = 2 * (X_hat - X) / X.shape[0]
            # Gradient w.r.t. decoder weights
            self.W_dec -= self.lr * Z.T @ grad_out
            self.b_dec -= self.lr * grad_out.sum(axis=0)
            return loss

        def fit(self, X_normal, epochs=300, verbose=True):
            '''Train on normal traffic. Set threshold at 99th percentile.'''
            for epoch in range(epochs):
                loss = self.train_step(X_normal)
                if verbose and epoch % 100 == 0:
                    print(f"  Epoch {epoch:3d}: MSE loss = {loss:.5f}")
            # Compute per-sample reconstruction errors on training set
            errors = [self.mse(X_normal[i:i+1], self.reconstruct(X_normal[i:i+1]))
                      for i in range(len(X_normal))]
            self.threshold = float(np.percentile(errors, 99))
            if verbose:
                print(f"  Threshold (99th pct): {self.threshold:.5f}")

        def score(self, x_raw: list) -> Tuple[float, bool]:
            '''Score a single feature vector. Returns (error, is_anomaly).'''
            x = np.array(x_raw, dtype=float).reshape(1, -1)
            error = self.mse(x, self.reconstruct(x))
            return error, (self.threshold is not None and error > self.threshold)

    # ── Feature vector builder ─────────────────────────────────────────────────
    def make_fv(bytes_out, bytes_in, pkts_out, pkts_in, flows_out,
                uniq_dst, uniq_port, pct_tcp, pct_udp, ent_ip, ent_port):
        '''Build normalized (0-1) 11-dimensional feature vector.'''
        return [bytes_out, bytes_in, pkts_out, pkts_in, flows_out,
                uniq_dst, uniq_port, pct_tcp, pct_udp, ent_ip, ent_port]

    # ── Generate normal training traffic ──────────────────────────────────────
    rng = np.random.default_rng(42)
    # Normal: balanced in/out, few destinations, high TCP, low entropy
    means = [0.30, 0.28, 0.31, 0.30, 0.30, 0.20, 0.10, 0.80, 0.12, 0.20, 0.15]
    stds  = [0.05, 0.04, 0.05, 0.05, 0.05, 0.04, 0.03, 0.05, 0.03, 0.04, 0.04]
    X_normal = np.clip(
        rng.normal(means, stds, size=(1200, 11)), 0.0, 1.0
    )

    print("=" * 60)
    print("AUTOENCODER — MULTI-DIMENSIONAL ANOMALY DETECTION")
    print("=" * 60)
    print(f"Training on {len(X_normal)} normal traffic windows (90 days x 5-min windows)...")
    ae = SimpleAutoencoder(input_dim=11, latent_dim=5, lr=0.008)
    ae.fit(X_normal, epochs=300, verbose=True)

    # ── Test scenarios ────────────────────────────────────────────────────────
    test_cases = [
        ("Normal window",
         make_fv(0.30, 0.29, 0.31, 0.30, 0.29, 0.21, 0.10, 0.81, 0.11, 0.21, 0.15)),

        ("DDoS (high outbound, high port entropy)",
         make_fv(0.95, 0.05, 0.97, 0.02, 0.94, 0.95, 0.90, 0.30, 0.65, 0.95, 0.95)),

        ("Data exfiltration (high outbound, rare destination)",
         make_fv(0.88, 0.08, 0.55, 0.04, 0.42, 0.05, 0.06, 0.72, 0.21, 0.82, 0.28)),

        ("Lateral movement (unusual dest IPs, low volume)",
         make_fv(0.25, 0.22, 0.28, 0.25, 0.60, 0.85, 0.75, 0.65, 0.25, 0.80, 0.78)),

        ("Subtle exfil (slightly high outbound + unusual destination)",
         make_fv(0.42, 0.15, 0.38, 0.12, 0.35, 0.12, 0.08, 0.74, 0.18, 0.55, 0.22)),
    ]

    print("\n[TEST RESULTS]")
    print(f"{'Scenario':<45} {'Error':>8} {'Thresh':>8} {'Result'}")
    print("-" * 75)

    for label, fv in test_cases:
        error, is_anom = ae.score(fv)
        verdict = "ANOMALY" if is_anom else "Normal "
        print(f"{label:<45} {error:8.5f} {ae.threshold:8.5f} [{verdict}]")
        if is_anom:
            # Find the most deviant features
            x = np.array(fv)
            x_hat = ae.reconstruct(x.reshape(1,-1))[0]
            deviations = [(FEATURES[i], abs(float(x[i]-x_hat[i])))
                          for i in range(len(FEATURES))]
            deviations.sort(key=lambda t: t[1], reverse=True)
            top_features = [f"{name}(dev={dev:.3f})" for name, dev in deviations[:3]]
            print(f"  Top deviating features: {', '.join(top_features)}")
            analysis = llm_explain(
                f"Autoencoder anomaly: reconstruction error={error:.5f} "
                f"(threshold={ae.threshold:.5f}). "
                f"Most deviant features: {top_features}. "
                f"Scenario: {label}. MITRE technique? Action? Under 60 words.",
                max_tokens=80
            )
            print(f"  LLM: {analysis}")

else:
    print("[SIMULATION] numpy not available - showing example output")
    print("  Normal window        error=0.00082  [Normal ]")
    print("  DDoS                 error=0.04219  [ANOMALY] -> MITRE T1498")
    print("  Data exfiltration    error=0.03751  [ANOMALY] -> MITRE T1048")
    print("  Lateral movement     error=0.02883  [ANOMALY] -> MITRE T1021")
    print("  Subtle exfil         error=0.01442  [ANOMALY] -> MITRE T1048")
    print("\nInstall numpy for full demo: pip install numpy")


## Demo 4: EWMA Trend Detector — Catching Slow-Burn Exfiltration

The hardest attack to detect: **the slow bleed**.

An attacker exfiltrating 100MB/hour for 10 hours never triggers a per-window z-score.
Each individual 5-minute window looks only slightly above normal.
Only by looking at the **trend over time** do you see the escalation.

**EWMA (Exponential Weighted Moving Average)** is how we track trends
without the complexity of a full LSTM neural network:

```
EWMA_t = alpha * observation + (1 - alpha) * EWMA_{t-1}

alpha = 0.1  ->  slow adaptation (long memory, catches slow trends)
alpha = 0.5  ->  fast adaptation (short memory, misses slow trends)
```

**Trend anomaly score** = how fast is the EWMA rising?
```
trend_score = (current_EWMA - baseline_EWMA) / baseline_std
```

If the trend score rises steadily across multiple windows even though no
single window breaks the z-score threshold — that is the slow exfiltration signal.

*Analogy: EWMA trend detection is like watching BGP route table growth.
A router adding 10 new routes per day is normal.
A router adding 100 new routes per day for 5 days straight —
each day within normal range — signals something is wrong with the topology.*

In [ ]:
# ── Demo 4: EWMA Trend Detector ──────────────────────────────────────────────

class EWMATrendDetector:
    '''
    Exponential Weighted Moving Average trend detector.
    Catches gradual anomalies that stay below per-window z-score thresholds.
    Ideal for slow-burn data exfiltration, gradual DDoS ramp-ups.
    '''

    def __init__(self, alpha: float = 0.15,
                 trend_threshold: float = 2.5,
                 warmup_windows: int = 12):
        self.alpha = alpha              # smoothing factor (0=no adaptation, 1=no memory)
        self.trend_thr = trend_threshold
        self.warmup = warmup_windows    # windows before alerts can fire

        self.ewma: Optional[float] = None
        self.ewma_sq: Optional[float] = None  # for online std computation
        self.baseline_ewma: Optional[float] = None
        self.baseline_std: Optional[float] = None
        self.window_count = 0
        self.history: List[Tuple[int, float, float]] = []  # (window, value, score)

    def update(self, value: float) -> Optional[dict]:
        '''
        Ingest one observation. Returns alert dict if trend anomaly detected.
        Uses Welford online algorithm for running mean/variance.
        '''
        self.window_count += 1

        # Initialize EWMA
        if self.ewma is None:
            self.ewma = value
            self.ewma_sq = value ** 2
        else:
            self.ewma = self.alpha * value + (1 - self.alpha) * self.ewma
            self.ewma_sq = self.alpha * (value**2) + (1 - self.alpha) * self.ewma_sq

        # EWMA-based variance: E[X^2] - E[X]^2
        ewma_var = max(self.ewma_sq - self.ewma**2, 1e-9)
        ewma_std = math.sqrt(ewma_var)

        # During warmup: establish baseline (first N windows are "normal")
        if self.window_count <= self.warmup:
            self.baseline_ewma = self.ewma
            self.baseline_std  = max(ewma_std, 1.0)
            self.history.append((self.window_count, value, 0.0))
            return None

        # Trend score: how far is current EWMA above baseline?
        trend_score = (self.ewma - self.baseline_ewma) / self.baseline_std
        self.history.append((self.window_count, value, trend_score))

        if trend_score >= self.trend_thr:
            return {
                "window": self.window_count,
                "current_value": value,
                "ewma": round(self.ewma, 1),
                "baseline_ewma": round(self.baseline_ewma, 1),
                "trend_score": round(trend_score, 3),
                "escalation_pct": round((self.ewma / self.baseline_ewma - 1) * 100, 1),
            }
        return None

# ── Scenario: slow exfiltration that stays below per-window z-score ───────────
print("=" * 65)
print("EWMA TREND DETECTOR — SLOW-BURN EXFILTRATION DETECTION")
print("=" * 65)

detector = EWMATrendDetector(alpha=0.15, trend_threshold=2.5, warmup_windows=12)

# Normal baseline: ~5 MB outbound per 5-minute window
random.seed(7)
normal_5min_mb = 5.0
print("\n[Phase 1: Warmup - 12 normal windows (1 hour)]")
for i in range(12):
    val = normal_5min_mb * random.uniform(0.85, 1.15)
    alert = detector.update(val)
    print(f"  Window {i+1:2d}: {val:.1f} MB -> EWMA={detector.ewma:.2f} [warmup]")

# Exfiltration: gradual increase, each window only slightly above normal
# Each window: +8% above previous. After 10 windows = 2.15x normal volume.
print("\n[Phase 2: Slow exfiltration begins - 20 windows (100 min)]")
print(f"{'Win':>4} {'Value':>8} {'EWMA':>8} {'TrendZ':>8} {'Alert'}")
print("-" * 45)

exfil_value = normal_5min_mb
alerts_fired = []
for i in range(20):
    exfil_value *= 1.08   # +8% per 5-min window
    # Add some noise to make it realistic
    noisy_val = exfil_value * random.uniform(0.92, 1.08)
    alert = detector.update(noisy_val)

    win_num = 13 + i
    trend_z = detector.history[-1][2] if detector.history else 0.0
    alert_str = " <-- TREND ALERT!" if alert else ""
    print(f"{win_num:4d} {noisy_val:8.1f} {detector.ewma:8.2f} {trend_z:8.3f}{alert_str}")

    if alert and alert not in alerts_fired:
        alerts_fired.append(alert)

if alerts_fired:
    first = alerts_fired[0]
    print(f"\n[ALERT FIRED at window {first['window']}]")
    print(f"  EWMA: {first['baseline_ewma']:.1f} MB (normal) -> {first['ewma']:.1f} MB (current)")
    print(f"  Escalation: +{first['escalation_pct']:.1f}% above baseline")
    print(f"  Trend Z-score: {first['trend_score']:.2f} (threshold: 2.5)")
    analysis = llm_explain(
        f"EWMA trend alert: gradual outbound traffic increase over "
        f"{first['window'] - 12} windows (5-min each). "
        f"Baseline was {first['baseline_ewma']:.0f} MB/window, "
        f"now {first['ewma']:.0f} MB/window (+{first['escalation_pct']:.0f}%). "
        f"Each individual window was below z-score threshold. "
        f"Slow exfiltration? MITRE T1048? Recommended action? Under 70 words.",
        max_tokens=90
    )
    print(f"  LLM: {analysis}")
    print(f"\n[EWMA] Per-window z-score would have seen: normal. "
          f"Trend detector caught it at window {first['window']}.")
else:
    print("\n[No trend alert - check threshold settings]")

print("\n[EWMA Detector] Best for: slow exfil, gradual DDoS ramp-up, data staging.")


## Demo 5: Full 3-Layer Fusion Pipeline

**All three detectors wired together** — the production-grade anomaly detection system.

```
NetFlow 5-min windows
        |
        +-> [Layer 1] Z-Score Baseline   -> volumetric spike alerts
        |
        +-> [Layer 2] Entropy Analyzer   -> distribution anomaly alerts
        |
        +-> [Layer 3] Autoencoder        -> multi-dimensional anomaly alerts
        |
        +-> [Layer 4] EWMA Trend         -> gradual escalation alerts
        |
        v
   Alert Fusion Engine
        |
        +-> Single layer hit   -> LOW confidence
        +-> Two layers hit     -> MEDIUM confidence
        +-> Three+ layers hit  -> HIGH confidence  -> LLM synthesis -> SOC queue
```

**Why multiple layers?** Each detector catches what the others miss:

| Attack | Z-Score | Entropy | Autoencoder | EWMA |
|--------|---------|---------|-------------|------|
| DDoS | YES (volume) | YES (source diversity) | YES | Possibly |
| Port scan | No (low volume) | YES (port entropy) | YES | No |
| Slow exfil | No (per-window) | No | Possibly | YES |
| Lateral move | No | YES (IP entropy) | YES | No |

> **An anomaly confirmed by 3 independent detectors has near-zero false positive rate.**

In [ ]:
# ── Demo 5: Full 3-Layer Fusion Pipeline ─────────────────────────────────────

@dataclass
class FusionAlert:
    '''Combined alert from multiple detection layers.'''
    alert_id: str
    device: str
    layers_triggered: List[str]
    confidence: str       # LOW / MEDIUM / HIGH
    combined_score: float
    z_score_max: float
    entropy_alerts: list
    ae_error: float
    ewma_trend: float
    llm_brief: str = ""

class FusionDetectionPipeline:
    '''
    Combines z-score, entropy, autoencoder, and EWMA into a single
    prioritized alert stream.
    '''

    def __init__(self, baseline: StratifiedBaseline,
                 entropy_analyzer: EntropyAnalyzer,
                 autoencoder=None):
        self.baseline  = baseline
        self.entropy   = entropy_analyzer
        self.ae        = autoencoder
        self.ewma_detectors: Dict[str, EWMATrendDetector] = {}
        self.alert_count = 0
        self.incidents: List[FusionAlert] = []

    def _new_id(self) -> str:
        self.alert_count += 1
        return f"ANOM-{self.alert_count:04d}"

    def _get_ewma(self, device: str) -> EWMATrendDetector:
        if device not in self.ewma_detectors:
            self.ewma_detectors[device] = EWMATrendDetector(
                alpha=0.15, trend_threshold=2.5, warmup_windows=8
            )
        return self.ewma_detectors[device]

    def analyze(self, window: TrafficWindow,
                flows: List[FlowRecord],
                fv: Optional[list] = None) -> Optional[FusionAlert]:
        '''
        Run all detectors on one 5-minute traffic window.
        Returns FusionAlert if at least one layer fires.
        '''
        layers = []
        scores = []

        # Layer 1: Z-Score
        z_result = self.baseline.score_window(window)
        z_max = z_result["max_z"]
        if z_max >= 2.5:
            layers.append("Z-SCORE")
            scores.append(min(z_max / 10.0, 1.0))

        # Layer 2: Entropy
        ent_result = self.entropy.analyze_flows(window.device, flows)
        entropy_alerts = ent_result["alerts"]
        if entropy_alerts:
            layers.append("ENTROPY")
            scores.append(0.7)

        # Layer 3: Autoencoder (if available)
        ae_error = 0.0
        if self.ae is not None and fv is not None and HAS_NUMPY:
            ae_error, ae_anom = self.ae.score(fv)
            if ae_anom:
                layers.append("AUTOENCODER")
                scores.append(min(ae_error / (self.ae.threshold * 3), 1.0))

        # Layer 4: EWMA trend
        ewma = self._get_ewma(window.device)
        ewma_alert = ewma.update(window.bytes_out)
        ewma_trend = ewma.history[-1][2] if ewma.history else 0.0
        if ewma_alert:
            layers.append("EWMA_TREND")
            scores.append(min(ewma_trend / 5.0, 1.0))

        if not layers:
            return None

        # Confidence by number of layers
        if len(layers) >= 3:
            confidence = "HIGH"
        elif len(layers) == 2:
            confidence = "MEDIUM"
        else:
            confidence = "LOW"

        combined = sum(scores) / len(scores) if scores else 0.0

        # LLM brief for MEDIUM/HIGH
        brief = ""
        if confidence in ("HIGH", "MEDIUM"):
            prompt = (
                f"Network anomaly fusion alert on device {window.device}:\n"
                f"Detection layers triggered: {layers}\n"
                f"Z-score max: {z_max:.1f}\n"
                f"Entropy alerts: {[a['type'] for a in entropy_alerts]}\n"
                f"Autoencoder error: {ae_error:.5f}\n"
                f"EWMA trend z-score: {ewma_trend:.2f}\n"
                f"Hour: {window.hour:02d}:00\n\n"
                f"Write 2-sentence incident brief: what attack type, "
                f"MITRE technique, immediate analyst action."
            )
            brief = llm_explain(prompt, max_tokens=100)

        alert = FusionAlert(
            alert_id=self._new_id(),
            device=window.device,
            layers_triggered=layers,
            confidence=confidence,
            combined_score=round(combined, 3),
            z_score_max=z_max,
            entropy_alerts=entropy_alerts,
            ae_error=ae_error,
            ewma_trend=ewma_trend,
            llm_brief=brief,
        )
        self.incidents.append(alert)
        return alert

# ── Build the pipeline ────────────────────────────────────────────────────────
print("=" * 65)
print("FULL 3-LAYER FUSION PIPELINE")
print("=" * 65)

# Warm up the EWMA (simulate 10 normal windows first)
pipeline = FusionDetectionPipeline(
    baseline=baseline,
    entropy_analyzer=analyzer,
    autoencoder=ae if HAS_NUMPY else None,
)
# Warm up EWMA for db-server-01
for _ in range(10):
    w_warm = TrafficWindow("db-server-01", 14, 1,
                           50_000_000 * random.uniform(0.9,1.1),
                           50_000, 10_000, 500)
    pipeline._get_ewma("db-server-01").update(w_warm.bytes_out)

# ── Test scenarios ────────────────────────────────────────────────────────────
print("\nAnalyzing 4 traffic scenarios...\n")

scenarios_5 = [
    # Scenario A: Normal
    {
        "label": "Normal business traffic (Mon 14:00)",
        "window": TrafficWindow("db-server-01", 14, 1, 52_000_000, 51_000, 10_200, 510),
        "flows": [FlowRecord("db-server-01","10.0.0.1",443,"tcp",50000)]*5,
        "fv": make_fv(0.30,0.29,0.31,0.30,0.29,0.21,0.10,0.81,0.11,0.21,0.15) if HAS_NUMPY else None,
    },
    # Scenario B: DDoS
    {
        "label": "DDoS attack (Sun 03:00 - 200x volume)",
        "window": TrafficWindow("db-server-01", 3, 6, 980_000_000, 950_000, 9_500, 48),
        "flows": [FlowRecord(f"10.100.{i}.1","db-server-01",80,"tcp",1000) for i in range(50)],
        "fv": make_fv(0.95,0.05,0.97,0.02,0.94,0.95,0.90,0.30,0.65,0.95,0.95) if HAS_NUMPY else None,
    },
    # Scenario C: Port scan (low volume but high entropy)
    {
        "label": "Port scan from external IP (low volume)",
        "window": TrafficWindow("db-server-01", 14, 1, 48_000_000, 49_500, 9_800, 512),
        "flows": [FlowRecord("185.220.1.100","db-server-01",p,"tcp",60)
                  for p in [22,23,25,53,80,110,135,139,443,445,1433,3389,5900,8080,8443]],
        "fv": make_fv(0.30,0.10,0.32,0.10,0.55,0.85,0.92,0.65,0.25,0.80,0.95) if HAS_NUMPY else None,
    },
    # Scenario D: Data exfiltration
    {
        "label": "Data exfiltration (high outbound to rare destination)",
        "window": TrafficWindow("db-server-01", 2, 6, 680_000_000, 12_000_000, 8_200, 3),
        "flows": [FlowRecord("db-server-01","91.200.1.5",443,"tcp",5_000_000)]*3,
        "fv": make_fv(0.88,0.08,0.55,0.04,0.42,0.05,0.06,0.72,0.21,0.82,0.28) if HAS_NUMPY else None,
    },
]

for sc in scenarios_5:
    alert = pipeline.analyze(sc["window"], sc["flows"], sc.get("fv"))
    if alert:
        conf_icon = {"HIGH":"[!!!]","MEDIUM":"[!! ]","LOW":"[!  ]"}.get(alert.confidence,"[   ]")
        print(f"{conf_icon} {sc['label']}")
        print(f"  ID={alert.alert_id} | Confidence={alert.confidence} | "
              f"Layers={alert.layers_triggered}")
        print(f"  z_max={alert.z_score_max:.1f} | ae_err={alert.ae_error:.5f} | "
              f"trend_z={alert.ewma_trend:.2f}")
        if alert.entropy_alerts:
            print(f"  Entropy: {[a['type'] for a in alert.entropy_alerts]}")
        if alert.llm_brief:
            print(f"  Brief: {alert.llm_brief}")
    else:
        print(f"[    ] {sc['label']} - no alert (all layers below threshold)")
    print()

print("=" * 65)
print(f"FUSION SUMMARY: {len(pipeline.incidents)} alerts | "
      f"HIGH={sum(1 for a in pipeline.incidents if a.confidence=='HIGH')} | "
      f"MEDIUM={sum(1 for a in pipeline.incidents if a.confidence=='MEDIUM')} | "
      f"LOW={sum(1 for a in pipeline.incidents if a.confidence=='LOW')}")
print("All HIGH-confidence alerts routed to SOC queue for human approval.")


## Summary: What You Built

A complete **3-layer network anomaly detection system** with fusion scoring:

| Layer | Method | Key Formula | Threshold |
|-------|--------|-------------|-----------|
| **Z-Score Baseline** | Stratified 168 time buckets | `z = (x - mean) / std` | z > 3.5 = alert |
| **Entropy Analyzer** | Shannon entropy | `H = -sum(p * log2(p))` | H > 0.60 = scan/tunnel |
| **Autoencoder** | 11-dim neural reconstruction | MSE(input, decoded) | > 99th pct of normal |
| **EWMA Trend** | Exponential moving average | `alpha*x + (1-alpha)*prev` | trend_z > 2.5 |
| **Fusion** | Layer vote count | 3 layers = HIGH confidence | Alert + LLM brief |

### What Each Layer Catches

```
DDoS attack:          Z-Score (volume spike) + Autoencoder (multi-dim) = HIGH
Port scan:            Entropy (port diversity) + Autoencoder            = MEDIUM/HIGH
Data exfiltration:    Z-Score + Autoencoder + Entropy (IP)              = HIGH
Slow exfiltration:    EWMA Trend (per-window below threshold)            = MEDIUM
DNS tunneling:        Entropy (query name length/diversity)              = MEDIUM
```

### Production Upgrade Path

```
Python rolling stats    ->  90-day stratified baseline per device per metric
Simple entropy          ->  NetFlow feature extraction at line rate (nProbe, GoFlow2)
numpy autoencoder       ->  PyTorch autoencoder retrained weekly on clean data
EWMA trend              ->  LSTM on 12-window sequences (1 hour lookback)
Manual fusion           ->  Apache Flink stream processing for real-time fusion
```

**Next: Chapter 80 — Securing AI Systems** — prompt injection, adversarial inputs,
model extraction attacks, and defensive architecture around your LLM components.
You built AI security systems. Now we secure the AI systems themselves.

In [ ]:
# ── Full Integration: Production Anomaly Detection Checklist ─────────────────
print("CHAPTER 75: PRODUCTION ANOMALY DETECTION CHECKLIST")
print("=" * 62)

checklist = [
    ("Data Sources",      "NetFlow v9 / IPFIX from all routers and core switches"),
    ("Data Sources",      "sFlow (1:1000 sampling) for 10G+ core links"),
    ("Data Sources",      "DNS query logs from recursive resolvers"),
    ("Data Sources",      "5-minute aggregation windows per source device"),
    ("Baseline",          "90 days minimum baseline before enabling alerts"),
    ("Baseline",          "168 time-slot buckets (24h x 7days) per metric"),
    ("Baseline",          "EWMA baseline update: alpha=0.02 (slow drift adaptation)"),
    ("Baseline",          "Re-baseline after major network changes or incidents"),
    ("Z-Score Thresholds","Low confidence:  z > 2.5 (log and monitor)"),
    ("Z-Score Thresholds","Medium:          z > 3.5 (analyst queue)"),
    ("Z-Score Thresholds","High confidence: z > 5.0 (immediate alert)"),
    ("Entropy",           "Port entropy > 0.60  ->  port scan (T1046)"),
    ("Entropy",           "IP entropy > 0.70    ->  IP scan or DDoS (T1046/T1498)"),
    ("Entropy",           "DNS subdomain len>20 ->  DNS tunnel (T1071.004)"),
    ("Autoencoder",       "Features: 11 metrics normalized 0-1 per device window"),
    ("Autoencoder",       "Architecture: 11 -> 5 latent -> 11 (relu/sigmoid)"),
    ("Autoencoder",       "Threshold: 99th percentile of normal reconstruction error"),
    ("Autoencoder",       "Retrain monthly; validate against known-normal held-out set"),
    ("Fusion",            "3 layers = HIGH confidence -> immediate SOC escalation"),
    ("Fusion",            "2 layers = MEDIUM -> analyst queue within 1 hour"),
    ("Fusion",            "1 layer  = LOW -> log, correlate with TI feed"),
    ("Guardrails",        "All blocking actions require analyst approval"),
    ("Guardrails",        "Monthly false-positive rate review -> threshold tuning"),
    ("Guardrails",        "Anomaly baseline drift check: retrain if FP rate > 5%"),
]

current = ""
for category, item in checklist:
    if category != current:
        print(f"\n[{category}]")
        current = category
    print(f"  + {item}")

print("\n" + "=" * 62)
print("3-LAYER DETECTION SUMMARY")
print("=" * 62)
print("Layer 1 Z-Score:   fast, interpretable, per-metric")
print("Layer 2 Entropy:   catches scans/tunneling without volume change")
print("Layer 3 AE:        multi-dimensional, catches subtle combinations")
print("Layer 4 EWMA:      catches slow trends invisible to per-window stats")
print("Layer 5 Fusion:    combines all -> confidence score -> LLM brief")
print("")
print("Each layer catches what the others miss.")
print("Three layers agreeing = near-zero false positive rate.")
print("Chapter 75 complete. Build the pipeline. Trust the patterns.")
